In [1]:
!apt update
!apt install unzip

Get:1 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2546 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1292 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6862 kB]33m
Get:9 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 Packages [266 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/ma

In [2]:
!unzip data.zip

Archive:  data.zip
   creating: train/
   creating: train/images/
  inflating: train/images/image58_jpg.rf.ec014bd39049bb7c0c48b58ea3e30d00.jpg  
  inflating: train/images/107_jpeg_jpg.rf.58b98abb95f08c6e2632335b94e98b8b.jpg  
  inflating: train/images/thumb_d_B80D8A0CBCCDADB88588D8C781E125FC_jpg.rf.7434c35f291b659e96cae2242064b1ea.jpg  
  inflating: train/images/241_jpeg_jpg.rf.6fcf8b0a84311e6bb42240f046e6f552.jpg  
  inflating: train/images/image27_jpg.rf.ef9de052e75fbbde0cddbd7987dc282c.jpg  
  inflating: train/images/8_jpeg_jpg.rf.7c32f02283a7dd831f01d414f97a916f.jpg  
  inflating: train/images/16723726-pedestrian-traffic-lights-green-light_jpg.rf.cadd48018aeb3ee08334380b925d05ec.jpg  
  inflating: train/images/201905151329337202_20_jpg.rf.74024241a1d730442dee1d1d8e31de60.jpg  
  inflating: train/images/008160_jpeg.rf.Tw4ghYQcDN87jQ2J7Tgn.jpeg  
  inflating: train/images/SSI_20211105170957_jpg.rf.db193e53fd45e20de507793748fcf13d.jpg  
  inflating: train/images/image67_jpg.rf.ab077b

In [3]:
!find . -type f -name "*Identifier*" -delete

In [ ]:
# 데이터셋 전부 제거
!rm -rf /workspace/train/labels/* /workspace/train/images/*
!rm -rf /workspace/valid/labels/* /workspace/valid/images/*

In [4]:
# 데이터셋 개수 확인
!ls /workspace/train/labels | wc -l
!ls /workspace/train/images | wc -l
!ls /workspace/valid/labels | wc -l
!ls /workspace/valid/images | wc -l

1619
1619
114
114


In [4]:
!pip install ultralytics
!pip install Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 224.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 243.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 200.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 221.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 236.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 238.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 284.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 235.9 MB/s eta 0:00:00
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 2.4.7
    Uninstalling pyparsing-2.4.7:
      Successfully uninstalled pyparsing-2.4.7
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:

In [ ]:
# 학습
from ultralytics import YOLO

# 1. 모델 로드 (v11 나노 모델)
model = YOLO('yolo11n.pt')

# 2. 학습 시작
results = model.train(
    data='data.yaml',
    epochs=150,
    imgsz=640,            # 640 고정 (원본 해상도와 찰떡궁합)
    batch=64,
    patience=30,          # 과적합 방지를 위한 조기 종료 (필수)
    
    # --- [핵심 수정] 원본 크기 훼손 방지 ---
    mosaic=0.1,           # 0.5 -> 0.1: 아예 0으로 끄거나 아주 가끔만 작동하게 만듭니다. (가장 중요)
    scale=0.0,            # 0.2 -> 0.0: 이미지를 축소/확대하지 말고 원본 크기 그대로 학습시킵니다.
    mixup=0.0,            # 이미지가 겹쳐서 흐려지는 현상 차단
    copy_paste=0.0,       # 모자이크를 끄면 copy_paste도 끄는 것이 충돌을 막습니다.
    
    # --- 분류 가중치 집중 ---
    box=10.0,
    cls=3.0,              # 분류 가중치는 높게 유지

    hsv_h=0.015,   # 빨강/초록의 정체성은 지키되 약간의 톤 변화만 허용
    hsv_s=0.7,     # 쨍한 불빛부터 빛바랜 불빛까지 모두 대응
    hsv_v=0.4,     # 역광이나 그늘진 환경 대응

    translate=0.1,  # 이미지 내에서 신호등의 위치를 상하좌우로 10% 정도 이동 (위치에 대한 과적합 방지)
    fliplr=0.5,     # 50% 확률로 좌우 반전 (신호등은 좌우가 뒤집혀도 정체성이 변하지 않으므로 데이터 2배 뻥튀기 효과)
    erasing=0.1,    # 10% 확률로 이미지의 일부를 가림 (나뭇가지나 표지판에 신호등이 살짝 가려지는 상황 대비)
)

WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.42 🚀 Python-3.11.10 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24210MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=Fa

In [6]:
!pip install openvino-dev

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 90.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 233.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
  Attempting uninstall: networkx
    Found existing installation: networkx 3.2.1
    Uninstalling networkx-3.2.1:
      Successfully uninstalled networkx-3.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update,

In [7]:
from ultralytics import YOLO

# 학습된 모델 로드
model = YOLO("runs/detect/train/weights/best.pt")

# OpenVINO 형식으로 저장 (정밀도 우선 설정)
model.export(
    format="openvino", 
    imgsz=640,       # 학습한 해상도와 동일하게 고정
    half=False,       # FP16 대신 FP32(Full Precision) 사용 (정확도 우선 시)
    int8=False,       # 정밀도가 깨질 수 있는 8비트 양자화는 사용 안 함
    dynamic=False     # 해상도를 고정하여 CPU 최적화 극대화
)

Ultralytics 8.4.42 🚀 Python-3.11.10 torch-2.4.1+cu124 CPU (AMD EPYC 75F3 32-Core Processor)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (5.2 MB)

OpenVINO: starting export with openvino 2024.6.0-17404-4c0f47d2335-releases/2024/6...
OpenVINO: export success ✅ 6.1s, saved as 'runs/detect/train/weights/best_openvino_model/' (10.2 MB)

Export complete (6.4s)
Results saved to /workspace/runs/detect/train/weights
Predict:         yolo predict task=detect model=runs/detect/train/weights/best_openvino_model/ imgsz=640 
Validate:        yolo val task=detect model=runs/detect/train/weights/best_openvino_model/ imgsz=640 data=data.yaml  
Visualize:       https://netron.app


'runs/detect/train/weights/best_openvino_model/'

In [7]:
!rm -rf runs